# Plot MERRA-2 Data by Region or State

This notebook demonstrates how to use the `region_mask.nc` file to filter and visualize MERRA-2 data for specific US regions or states.

**Features:**
- Load region and state masks aligned to MERRA-2 grid
- Filter data by region (1-10) or individual state
- Visualize spatial patterns for selected areas
- Calculate regional statistics

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import warnings
warnings.filterwarnings('ignore')

# Import standard plotting configuration
import plot_config as pc

%matplotlib inline

## 1. Load Region/State Masks

In [ ]:
# Load the mask file
mask_file = 'masks/region_mask.nc'
mask_ds = xr.open_dataset(mask_file)

print("Mask Dataset:")
print(mask_ds)
print("\nAvailable variables:")
for var in mask_ds.data_vars:
    print(f"  - {var}: {mask_ds[var].attrs.get('long_name', 'No description')}")

## 2. Explore Region and State Metadata

In [ ]:
# Display region information
print("10 US Regions:")
print("=" * 70)
for i, region_id in enumerate(mask_ds.region_id.values):
    region_name = mask_ds.region_name.values[i]
    # Find states in this region
    states_in_region = mask_ds.state_abbr.values[mask_ds.state_region.values == region_id]
    # Decode bytes to strings
    states_str = ', '.join([s.decode() if isinstance(s, bytes) else s for s in states_in_region])
    pixel_count = np.sum(mask_ds.region_mask.values == region_id)
    print(f"Region {region_id:2d}: {region_name:20s} | {pixel_count:4d} pixels | {states_str}")

In [ ]:
# Display state information (first 10 states as example)
print("\nState Information (first 10):")
print("=" * 70)
for i in range(min(10, len(mask_ds.state_id.values))):
    state_id = mask_ds.state_id.values[i]
    state_abbr = mask_ds.state_abbr.values[i]
    state_abbr_str = state_abbr.decode() if isinstance(state_abbr, bytes) else state_abbr
    state_name = mask_ds.state_name.values[i]
    region_id = mask_ds.state_region.values[i]
    region_name = mask_ds.region_name.values[region_id - 1]
    pixel_count = np.sum(mask_ds.state_mask.values == state_id)
    print(f"{state_abbr_str} ({state_name:14s}): {pixel_count:3d} pixels | Region {region_id} ({region_name})")

## 3. Load Sample MERRA-2 Data

Load a sample day of MERRA-2 data to demonstrate filtering and plotting.

In [ ]:
# Load sample MERRA-2 file (Temperature and VPD)
sample_date = '20200615'  # June 15, 2020 - summer day
merra2_file = f'daily_data/merra2_us_{sample_date}.nc'

# Load data
data_ds = xr.open_dataset(merra2_file)

print(f"MERRA-2 Data for {sample_date}:")
print(data_ds)

# Calculate daily mean for plotting (average over time dimension)
t2m_daily = data_ds.T2M.mean(dim='time')
vpd_daily = data_ds.VPD.mean(dim='time')

print(f"\nDaily mean T2M range: {t2m_daily.min().values:.1f} to {t2m_daily.max().values:.1f} °C")
print(f"Daily mean VPD range: {vpd_daily.min().values:.2f} to {vpd_daily.max().values:.2f} kPa")

## 4. Visualize Region Mask

In [ ]:
# Plot the region mask
fig = plt.figure(figsize=(14, 8))
ax = plt.axes(projection=ccrs.PlateCarree())

# Plot region mask with grid edges (Method 5)
im = ax.pcolormesh(
    mask_ds.lon, mask_ds.lat, mask_ds.region_mask,
    cmap='tab10', vmin=0.5, vmax=10.5,
    edgecolors='white', linewidth=0.2, alpha=0.95,
    transform=ccrs.PlateCarree()
)

# Add map features
ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='black')
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.coastlines()

# Set extent
ax.set_extent([-125, -66, 24, 49], crs=ccrs.PlateCarree())

# Add gridlines
gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False

# Colorbar
cbar = plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.05, shrink=0.7)
cbar.set_label('Region ID', fontsize=12)
cbar.set_ticks(range(1, 11))

plt.title('10 US Regions - MERRA-2 Grid', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Filter and Plot Data by Region

Example: Filter data for **Region 6 (South Central: AR, LA, NM, OK, TX)**

In [ ]:
# Select region
region_id = 6
region_name = mask_ds.region_name.values[region_id - 1]
region_states = mask_ds.state_abbr.values[mask_ds.state_region.values == region_id]

print(f"Selected Region: {region_id} - {region_name}")
# Decode bytes to strings for printing
states_str = ', '.join([s.decode() if isinstance(s, bytes) else s for s in region_states])
print(f"States: {states_str}")

# Filter data using xarray.where()
t2m_region = t2m_daily.where(mask_ds.region_mask == region_id)
vpd_region = vpd_daily.where(mask_ds.region_mask == region_id)

# Calculate statistics for the region
print(f"\nRegion {region_id} Statistics (Daily Mean):")
print(f"  Temperature: {t2m_region.mean().values:.2f} ± {t2m_region.std().values:.2f} °C")
print(f"  VPD: {vpd_region.mean().values:.2f} ± {vpd_region.std().values:.2f} kPa")
print(f"  Pixels in region: {np.sum(~np.isnan(t2m_region.values))}")

In [ ]:
# Plot filtered data for Region 6
fig, axes = plt.subplots(1, 2, figsize=(16, 6), subplot_kw={'projection': ccrs.PlateCarree()})

# Plot 1: Temperature (with grid edges)
ax = axes[0]
im1 = ax.pcolormesh(
    data_ds.lon, data_ds.lat, t2m_region,
    cmap='RdYlBu_r', vmin=15, vmax=35,
    edgecolors='white', linewidth=0.2, alpha=0.95,
    transform=ccrs.PlateCarree()
)
ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='black')
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.set_extent([-125, -66, 24, 49])
gl1 = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
gl1.top_labels = False
gl1.right_labels = False
cbar1 = plt.colorbar(im1, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8)
cbar1.set_label('Temperature (°C)', fontsize=11)
ax.set_title(f'Temperature - Region {region_id} ({region_name})\n{sample_date}', fontsize=12, fontweight='bold')

# Plot 2: VPD (with grid edges)
ax = axes[1]
im2 = ax.pcolormesh(
    data_ds.lon, data_ds.lat, vpd_region,
    cmap='YlOrRd', vmin=0, vmax=4,
    edgecolors='white', linewidth=0.2, alpha=0.95,
    transform=ccrs.PlateCarree()
)
ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='black')
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.set_extent([-125, -66, 24, 49])
gl2 = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
gl2.top_labels = False
gl2.right_labels = False
cbar2 = plt.colorbar(im2, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8)
cbar2.set_label('VPD (kPa)', fontsize=11)
ax.set_title(f'VPD - Region {region_id} ({region_name})\n{sample_date}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Filter and Plot Data by State

Example: Filter data for **Texas (TX)**

In [ ]:
# Select state
state_abbr = 'TX'

# Find state ID from abbreviation (compare with bytes)
state_abbr_bytes = state_abbr.encode() if isinstance(state_abbr, str) else state_abbr
state_idx = np.where(mask_ds.state_abbr.values == state_abbr_bytes)[0][0]
state_id = mask_ds.state_id.values[state_idx]
state_name = mask_ds.state_name.values[state_idx]
state_region_id = mask_ds.state_region.values[state_idx]
state_region_name = mask_ds.region_name.values[state_region_id - 1]

print(f"Selected State: {state_abbr} ({state_name})")
print(f"  State ID: {state_id}")
print(f"  Region: {state_region_id} ({state_region_name})")

# Filter data using xarray.where()
t2m_state = t2m_daily.where(mask_ds.state_mask == state_id)
vpd_state = vpd_daily.where(mask_ds.state_mask == state_id)

# Calculate statistics for the state
print(f"\n{state_abbr} Statistics (Daily Mean):")
print(f"  Temperature: {t2m_state.mean().values:.2f} ± {t2m_state.std().values:.2f} °C")
print(f"  VPD: {vpd_state.mean().values:.2f} ± {vpd_state.std().values:.2f} kPa")
print(f"  Pixels in state: {np.sum(~np.isnan(t2m_state.values))}")

In [ ]:
# Plot filtered data for Texas
fig, axes = plt.subplots(1, 2, figsize=(16, 6), subplot_kw={'projection': ccrs.PlateCarree()})

# Plot 1: Temperature (with grid edges)
ax = axes[0]
im1 = ax.pcolormesh(
    data_ds.lon, data_ds.lat, t2m_state,
    cmap='RdYlBu_r', vmin=15, vmax=35,
    edgecolors='white', linewidth=0.2, alpha=0.95,
    transform=ccrs.PlateCarree()
)
ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='black')
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.set_extent([-125, -66, 24, 49])
gl1 = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
gl1.top_labels = False
gl1.right_labels = False
cbar1 = plt.colorbar(im1, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8)
cbar1.set_label('Temperature (°C)', fontsize=11)
ax.set_title(f'Temperature - {state_name} ({state_abbr})\n{sample_date}', fontsize=12, fontweight='bold')

# Plot 2: VPD (with grid edges)
ax = axes[1]
im2 = ax.pcolormesh(
    data_ds.lon, data_ds.lat, vpd_state,
    cmap='YlOrRd', vmin=0, vmax=4,
    edgecolors='white', linewidth=0.2, alpha=0.95,
    transform=ccrs.PlateCarree()
)
ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='black')
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.set_extent([-125, -66, 24, 49])
gl2 = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
gl2.top_labels = False
gl2.right_labels = False
cbar2 = plt.colorbar(im2, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8)
cbar2.set_label('VPD (kPa)', fontsize=11)
ax.set_title(f'VPD - {state_name} ({state_abbr})\n{sample_date}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Zoom into Selected State

Create a zoomed-in view of the selected state showing only the pixels within the state boundary.

In [ ]:
# Find bounding box for the selected state
state_lons = data_ds.lon.values[np.any(mask_ds.state_mask == state_id, axis=0)]
state_lats = data_ds.lat.values[np.any(mask_ds.state_mask == state_id, axis=1)]

if len(state_lons) > 0 and len(state_lats) > 0:
    lon_min, lon_max = state_lons.min() - 1, state_lons.max() + 1
    lat_min, lat_max = state_lats.min() - 1, state_lats.max() + 1
    
    print(f"{state_abbr} Bounding Box:")
    print(f"  Longitude: {lon_min:.2f} to {lon_max:.2f}")
    print(f"  Latitude: {lat_min:.2f} to {lat_max:.2f}")
    
    # Plot zoomed view
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), subplot_kw={'projection': ccrs.PlateCarree()})
    
    # Plot 1: Temperature (zoomed, with grid edges)
    ax = axes[0]
    im1 = ax.pcolormesh(
        data_ds.lon, data_ds.lat, t2m_state,
        cmap='RdYlBu_r', vmin=20, vmax=35,
        edgecolors='white', linewidth=0.2, alpha=0.95,
        transform=ccrs.PlateCarree()
    )
    ax.add_feature(cfeature.STATES, linewidth=1.0, edgecolor='black')
    ax.add_feature(cfeature.COASTLINE, linewidth=1.0)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max])
    gl1 = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
    gl1.top_labels = False
    gl1.right_labels = False
    cbar1 = plt.colorbar(im1, ax=ax, pad=0.05, shrink=0.8)
    cbar1.set_label('Temperature (°C)', fontsize=10)
    ax.set_title(f'{state_abbr} Temperature (Zoomed)\n{sample_date}', fontsize=11, fontweight='bold')
    
    # Plot 2: VPD (zoomed, with grid edges)
    ax = axes[1]
    im2 = ax.pcolormesh(
        data_ds.lon, data_ds.lat, vpd_state,
        cmap='YlOrRd', vmin=0, vmax=4,
        edgecolors='white', linewidth=0.2, alpha=0.95,
        transform=ccrs.PlateCarree()
    )
    ax.add_feature(cfeature.STATES, linewidth=1.0, edgecolor='black')
    ax.add_feature(cfeature.COASTLINE, linewidth=1.0)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max])
    gl2 = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
    gl2.top_labels = False
    gl2.right_labels = False
    cbar2 = plt.colorbar(im2, ax=ax, pad=0.05, shrink=0.8)
    cbar2.set_label('VPD (kPa)', fontsize=10)
    ax.set_title(f'{state_abbr} VPD (Zoomed)\n{sample_date}', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
else:
    print(f"No pixels found for state {state_abbr}")

## 8. Interactive State/Region Selection

Modify the cells below to explore different states or regions.

In [ ]:
# ==== MODIFY THIS CELL TO CHANGE SELECTION ====

# Option 1: Select by state abbreviation
selected_state = 'CA'  # Change this to any state: 'TX', 'FL', 'NY', etc.

# Option 2: Select by region ID (comment out state selection above to use this)
# selected_region = 9  # Change this to any region: 1-10

# ==== END MODIFICATION ====

# Apply selection
if 'selected_state' in locals():
    # Filter by state (compare with bytes)
    selected_state_bytes = selected_state.encode() if isinstance(selected_state, str) else selected_state
    state_idx = np.where(mask_ds.state_abbr.values == selected_state_bytes)[0]
    if len(state_idx) > 0:
        state_id = mask_ds.state_id.values[state_idx[0]]
        state_name = mask_ds.state_name.values[state_idx[0]]
        filtered_t2m = t2m_daily.where(mask_ds.state_mask == state_id)
        filtered_vpd = vpd_daily.where(mask_ds.state_mask == state_id)
        title_suffix = f"{state_name} ({selected_state})"
        print(f"Selected: {title_suffix}")
        print(f"  Pixels: {np.sum(~np.isnan(filtered_t2m.values))}")
        print(f"  Mean T2M: {filtered_t2m.mean().values:.2f} °C")
        print(f"  Mean VPD: {filtered_vpd.mean().values:.2f} kPa")
    else:
        print(f"State '{selected_state}' not found in mask.")
        filtered_t2m = None
elif 'selected_region' in locals():
    # Filter by region
    if 1 <= selected_region <= 10:
        region_name = mask_ds.region_name.values[selected_region - 1]
        region_states = mask_ds.state_abbr.values[mask_ds.state_region.values == selected_region]
        # Decode bytes to strings for printing
        states_str = ', '.join([s.decode() if isinstance(s, bytes) else s for s in region_states])
        filtered_t2m = t2m_daily.where(mask_ds.region_mask == selected_region)
        filtered_vpd = vpd_daily.where(mask_ds.region_mask == selected_region)
        title_suffix = f"Region {selected_region} ({region_name})"
        print(f"Selected: {title_suffix}")
        print(f"  States: {states_str}")
        print(f"  Pixels: {np.sum(~np.isnan(filtered_t2m.values))}")
        print(f"  Mean T2M: {filtered_t2m.mean().values:.2f} °C")
        print(f"  Mean VPD: {filtered_vpd.mean().values:.2f} kPa")
    else:
        print(f"Region {selected_region} is out of range (1-10).")
        filtered_t2m = None
else:
    print("No selection specified. Set 'selected_state' or 'selected_region' above.")
    filtered_t2m = None

In [ ]:
# Plot the selected area
if filtered_t2m is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), subplot_kw={'projection': ccrs.PlateCarree()})
    
    # Plot 1: Temperature (with grid edges)
    ax = axes[0]
    im1 = ax.pcolormesh(
        data_ds.lon, data_ds.lat, filtered_t2m,
        cmap='RdYlBu_r', vmin=15, vmax=35,
        edgecolors='white', linewidth=0.2, alpha=0.95,
        transform=ccrs.PlateCarree()
    )
    ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='black')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.set_extent([-125, -66, 24, 49])
    gl1 = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
    gl1.top_labels = False
    gl1.right_labels = False
    cbar1 = plt.colorbar(im1, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8)
    cbar1.set_label('Temperature (°C)', fontsize=11)
    ax.set_title(f'Temperature - {title_suffix}\n{sample_date}', fontsize=12, fontweight='bold')
    
    # Plot 2: VPD (with grid edges)
    ax = axes[1]
    im2 = ax.pcolormesh(
        data_ds.lon, data_ds.lat, filtered_vpd,
        cmap='YlOrRd', vmin=0, vmax=4,
        edgecolors='white', linewidth=0.2, alpha=0.95,
        transform=ccrs.PlateCarree()
    )
    ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='black')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.set_extent([-125, -66, 24, 49])
    gl2 = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
    gl2.top_labels = False
    gl2.right_labels = False
    cbar2 = plt.colorbar(im2, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8)
    cbar2.set_label('VPD (kPa)', fontsize=11)
    ax.set_title(f'VPD - {title_suffix}\n{sample_date}', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

## 9. Summary: Using the Mask in Your Analysis

### Basic Usage Patterns

```python
# Load mask
mask_ds = xr.open_dataset('masks/region_mask.nc')

# Filter by region (e.g., Region 4 = Southeast)
data_southeast = data.where(mask_ds.region_mask == 4)

# Filter by state (e.g., Texas)
tx_id = mask_ds.state_id.values[mask_ds.state_abbr.values == 'TX'][0]
data_texas = data.where(mask_ds.state_mask == tx_id)

# Calculate regional statistics
regional_mean = data_southeast.mean(dim=['lat', 'lon'])
regional_std = data_southeast.std(dim=['lat', 'lon'])

# Group by region for multi-region analysis
for region_id in range(1, 11):
    region_data = data.where(mask_ds.region_mask == region_id)
    # Perform analysis...
```

### Available Mask Variables

- `region_mask`: (lat, lon) array with values 0-10 (0=non-US, 1-10=regions)
- `state_mask`: (lat, lon) array with values 0-50 (0=non-US, 1-50=states)
- `region_id`, `region_name`: Lookup tables for regions
- `state_id`, `state_abbr`, `state_name`, `state_region`: Lookup tables for states

In [ ]:
# Close datasets
mask_ds.close()
data_ds.close()

print("Done!")